# SPY Risk Alert Project Pipeline

This cumulative pipeline covers Stage04 ingestion, Stage05 storage, Stage06 preprocessing, Stage07 outlier analysis, and Stage08 EDA. It defaults to retained raw data; set `REFRESH_RAW = True` only for an intentional new Nasdaq acquisition. Stage07 flags and retains extreme returns; Stage08 produces descriptive summaries only, not a future target or trading recommendation.

## 1. Project Root, Configuration, and Imports

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys

if Path.cwd().name == 'notebooks':
    os.chdir('..')  # project/notebooks -> project
ROOT = Path.cwd()
if not (ROOT / 'src' / 'ingestion.py').is_file():
    for candidate in (ROOT, *ROOT.parents):
        project_candidate = candidate / 'project'
        if (project_candidate / 'src' / 'ingestion.py').is_file():
            ROOT = project_candidate
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datetime import date

from src.cleaning import clean_spy_ohlcv, write_cleaning_report
from src.eda import eda_summary, prepare_spy_eda_frame
from src.outliers import (
    analyze_daily_return_outliers,
    summarize_return_sensitivity,
    write_outlier_report,
)
from src.config import get_processed_data_dir, get_raw_data_dir, load_env
from src.ingestion import (
    fetch_nasdaq_history, timestamp_utc, validate_spy_history, write_manifest, write_raw_csv
)
from src.storage import get_parquet_engine, read_df, validate_roundtrip, write_df

environment_loaded = load_env()
RAW_DIR = get_raw_data_dir()
PROCESSED_DIR = get_processed_data_dir()
print('working from:', ROOT.name)
print('Environment loaded:', environment_loaded)
print('Raw directory:', RAW_DIR)
print('Processed directory:', PROCESSED_DIR)

working from: project
Environment loaded: True
Raw directory: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/data/raw
Processed directory: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/data/processed


## 2. Stage04 Raw Snapshot

Normal runs reuse the latest retained raw CSV. An explicit refresh validates and saves a new raw snapshot plus manifest.

In [2]:
REFRESH_RAW = False
SYMBOL = 'SPY'
csv_schema = {
    'open': 'float64', 'high': 'float64', 'low': 'float64',
    'close': 'float64', 'volume': 'int64',
}
raw_candidates = sorted(RAW_DIR.glob('api_nasdaq_spy_daily_*.csv'))

if REFRESH_RAW:
    end_date = date.today()
    start_date = end_date.replace(year=end_date.year - 10)
    spy_raw, source_metadata = fetch_nasdaq_history(
        SYMBOL, start_date=start_date.isoformat(), end_date=end_date.isoformat()
    )
    validation = validate_spy_history(spy_raw)
    snapshot_timestamp = timestamp_utc()
    raw_path = write_raw_csv(
        spy_raw, RAW_DIR, 'api_nasdaq_spy_daily', timestamp=snapshot_timestamp
    )
    manifest_path = write_manifest(
        {
            'path': raw_path, 'dataset': 'SPY daily unadjusted OHLCV',
            'rows': len(spy_raw), 'columns': list(spy_raw.columns),
            'source_metadata': source_metadata, 'validation': validation,
        },
        RAW_DIR / f'ingestion_manifest_{snapshot_timestamp}.json',
    )
    print('Refreshed raw snapshot:', raw_path.name)
    print('Saved manifest:', manifest_path.name)
else:
    if not raw_candidates:
        raise FileNotFoundError('No Stage04 SPY raw snapshot found. Set REFRESH_RAW = True.')
    raw_path = raw_candidates[-1]
    snapshot_timestamp = raw_path.stem.removeprefix('api_nasdaq_spy_daily_')
    spy_raw = read_df(raw_path, parse_dates=['date'], dtype=csv_schema)
    validation = validate_spy_history(spy_raw)
    print('Reused raw snapshot:', raw_path.name)

print('Rows and columns:', validation['shape'])
print('Date range:', validation['date_min'], 'to', validation['date_max'])
spy_raw.head()

Reused raw snapshot: api_nasdaq_spy_daily_20260907-143336.csv
Rows and columns: [2512, 6]
Date range: 2016-09-07 to 2026-09-04


,date,open,high,low,close,volume
0,2016-09-07,218.84,219.2200,218.30,219.01,76302150
1,2016-09-08,218.62,218.9400,218.15,218.51,73855230
2,2016-09-09,216.97,217.0300,213.25,213.28,220309300
3,2016-09-12,212.39,216.8100,212.31,216.34,167653400
4,2016-09-13,214.84,215.1499,212.50,213.23,182323200


## 3. Stage05 Typed Storage

The Parquet file is a typed representation of the named raw snapshot, not a cleaning or feature step.

In [3]:
storage_path = PROCESSED_DIR / f'spy_ohlcv_nasdaq_{snapshot_timestamp}.parquet'
write_df(spy_raw, storage_path)
spy_stored = read_df(storage_path)
storage_roundtrip = validate_roundtrip(
    spy_raw, spy_stored,
    {'date': 'datetime', 'open': 'float', 'close': 'float', 'volume': 'integer'},
)
if not storage_roundtrip['passed']:
    raise ValueError(f'Stage05 storage validation failed: {storage_roundtrip}')
print('Parquet engine:', get_parquet_engine())
print('Stored file:', storage_path.name)
print('Storage round-trip passed:', storage_roundtrip['passed'])

Parquet engine: pyarrow
Stored file: spy_ohlcv_nasdaq_20260907-143336.parquet
Storage round-trip passed: True


## 4. Stage06 Deterministic Preprocessing

The policy reparses and validates canonical OHLCV fields, sorts dates, and records its non-actions. It does not impute prices, remove outliers, or fit a global scaler.

In [4]:
spy_clean, cleaning_report = clean_spy_ohlcv(spy_stored)
preprocessed_path = PROCESSED_DIR / f'spy_ohlcv_preprocessed_{snapshot_timestamp}.parquet'
report_path = PROCESSED_DIR / f'cleaning_report_{snapshot_timestamp}.json'
write_df(spy_clean, preprocessed_path)
write_cleaning_report(cleaning_report, report_path)
spy_reloaded = read_df(preprocessed_path)
preprocessing_roundtrip = validate_roundtrip(
    spy_clean, spy_reloaded,
    {'date': 'datetime', 'open': 'float', 'close': 'float', 'volume': 'integer'},
)
if not preprocessing_roundtrip['passed']:
    raise ValueError(f'Stage06 preprocessing validation failed: {preprocessing_roundtrip}')

print('Preprocessed file:', preprocessed_path.name)
print('Cleaning report:', report_path.name)
print('Rows dropped:', cleaning_report['rows_dropped'])
print('Preprocessing round-trip passed:', preprocessing_roundtrip['passed'])

Preprocessed file: spy_ohlcv_preprocessed_20260907-143336.parquet
Cleaning report: cleaning_report_20260907-143336.json
Rows dropped: 0
Preprocessing round-trip passed: True


## 5. Stage07 Return Outlier Analysis

Daily close-to-close returns are flagged with IQR (`k=1.5`) and z-score (`|z| > 3`) rules. All market dates are retained. Filtered and 5%/95% winsorized variants are sensitivity diagnostics, not production transformations.


In [5]:
spy_return_analysis, outlier_report = analyze_daily_return_outliers(spy_clean)
sensitivity_summary = summarize_return_sensitivity(spy_return_analysis)

flags_path = PROCESSED_DIR / f'spy_return_outlier_flags_{snapshot_timestamp}.parquet'
sensitivity_path = PROCESSED_DIR / f'return_outlier_sensitivity_{snapshot_timestamp}.csv'
outlier_report_path = PROCESSED_DIR / f'outlier_analysis_report_{snapshot_timestamp}.json'
write_df(spy_return_analysis, flags_path)
write_df(sensitivity_summary, sensitivity_path)
write_outlier_report(outlier_report, outlier_report_path)

flags_reloaded = read_df(flags_path)
sensitivity_reloaded = read_df(sensitivity_path)
flags_roundtrip = validate_roundtrip(
    spy_return_analysis,
    flags_reloaded,
    {'date': 'datetime', 'close': 'float', 'daily_return': 'float', 'volume': 'integer'},
)
sensitivity_roundtrip = validate_roundtrip(
    sensitivity_summary,
    sensitivity_reloaded,
    {'observations': 'integer', 'mean': 'float', 'std': 'float'},
)
if not flags_roundtrip['passed'] or not sensitivity_roundtrip['passed']:
    raise ValueError('Stage07 storage validation failed')

print('Outlier flags:', flags_path.name)
print('Sensitivity table:', sensitivity_path.name)
print('IQR flags retained:', outlier_report['iqr']['flagged_count'])
print('Z-score flags retained:', outlier_report['zscore']['flagged_count'])
print('Outlier round-trips passed:', flags_roundtrip['passed'] and sensitivity_roundtrip['passed'])
display(sensitivity_summary)


Outlier flags: spy_return_outlier_flags_20260907-143336.parquet
Sensitivity table: return_outlier_sensitivity_20260907-143336.csv
IQR flags retained: 165
Z-score flags retained: 33
Outlier round-trips passed: True


,variant,observations,mean,median,std,minimum,maximum
0,all,2511,0.000566,0.000699,0.011363,-0.109424,0.105019
1,filtered_iqr,2346,0.001043,0.000848,0.007344,-0.018158,0.019981
2,winsorized_0.05_0.95,2511,0.000642,0.000699,0.008169,-0.016792,0.015605


## 6. Stage08 Exploratory Data Analysis

This summary profiles retained Stage07 observations and EDA-only return/range/volume/rolling-volatility fields. It records descriptive patterns and attention flags without treating correlation as causation or full-sample values as deployment rules.


In [6]:
spy_eda = prepare_spy_eda_frame(spy_return_analysis)
eda_tables = eda_summary(spy_eda)
eda_paths = {
    name: PROCESSED_DIR / f"spy_eda_{name}_{snapshot_timestamp}.csv"
    for name in eda_tables
}
for name, table in eda_tables.items():
    write_df(table, eda_paths[name])

for name, table in eda_tables.items():
    reloaded = read_df(eda_paths[name])
    if len(reloaded) != len(table) or list(reloaded.columns) != list(table.columns):
        raise ValueError(f"Stage08 EDA table validation failed for {name}")

attention_records = len(eda_tables["attention"])
print("EDA tables:", ", ".join(path.name for path in eda_paths.values()))
print("EDA rows retained:", len(spy_eda))
print("Structural missing cells:", int(spy_eda.isna().sum().sum()))
print("Attention records:", attention_records)
display(eda_tables["attention"])


EDA tables: spy_eda_overview_20260907-143336.csv, spy_eda_column_profile_20260907-143336.csv, spy_eda_numeric_summary_20260907-143336.csv, spy_eda_categorical_summary_20260907-143336.csv, spy_eda_datetime_summary_20260907-143336.csv, spy_eda_attention_20260907-143336.csv
EDA rows retained: 2512
Structural missing cells: 23
Attention records: 4


,column,role,missing_count,missing_fraction,dominant_fraction,attention
0,daily_return,numeric,1,0.000398,0.001990,has_missing
1,return_outlier_zscore,categorical,0,0.000000,0.986863,dominant_category
2,abs_return,numeric,1,0.000398,0.001990,has_missing
3,rolling_volatility_21,numeric,21,0.008360,0.008360,has_missing


## Sources, Storage, Preprocessing, Outliers, EDA, Assumptions, and Risks

- Source rules: `docs/data_sources.md`; storage lineage: `docs/data_storage.md`; preprocessing policy: `docs/preprocessing.md`; outlier policy: `docs/outliers.md`; EDA policy: `docs/eda.md`.
- `data/raw/` remains immutable; processed outputs are reproducible from the named raw snapshot and code.
- Stage07 flags and retains extreme returns. Stage08 documents full-snapshot descriptive patterns but does not define a production threshold, target, or causal relationship.
- Future-target definition, train-only transforms/thresholds, feature engineering, chronological modeling, and reporting remain later decisions.
